In [ ]:
!pip install kaggle

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!mkdir -p ~/.kaggle
!cp "/content/drive/MyDrive/MyDrive kaggle/kaggle.json" ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d blastchar/telco-customer-churn

Dataset URL: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
License(s): copyright-authors
  0% 0.00/172k [00:00<?, ?B/s]
100% 172k/172k [00:00<00:00, 120MB/s]


In [ ]:
from zipfile import ZipFile
dataset="/content/telco-customer-churn.zip"
with ZipFile(dataset,"r") as zip:
  zip.extractall()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import seaborn as sns

In [ ]:
telecom=pd.read_csv("/content/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [ ]:
telecom.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
telecom.isnull().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


Feature Engineering



In [ ]:
telecom["tenure_group"]=pd.cut(telecom["tenure"],bins=[0,6,24,100],labels=["New","Medium","Long term"],include_lowest=True)

In [ ]:
telecom["TotalCharges"]=pd.to_numeric(telecom["TotalCharges"],errors="coerce")

In [ ]:
telecom.fillna({"TotalCharges":0},inplace=True)

In [ ]:
telecom["monthly_avg"]=telecom["TotalCharges"]/telecom["tenure"]

In [ ]:
telecom["is_autopay"]=telecom["PaymentMethod"].apply(lambda x:1 if x in ["Bank transfer","Credit card"] else 0)

In [ ]:
print(telecom.columns)

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn',
       'tenure_group', 'monthly_avg', 'is_autopay'],
      dtype='object')


In [ ]:
service_cols=["PhoneService","MultipleLines","OnlineSecurity","OnlineBackup","DeviceProtection","TechSupport","StreamingTV","StreamingMovies"]
telecom["num_services"]=(telecom[service_cols]=="Yes").sum(axis=1)

In [ ]:
telecom["short_contract"]=telecom["Contract"].apply(lambda x:1 if x=="Month-to-Month" else 0)

In [ ]:
telecom["is_fiber"]=telecom["InternetService"].apply(lambda x:1 if x=="Fiber optic" else 0)

In [ ]:
telecom["tech_risk"]=((telecom["InternetService"]!="None")&(telecom["TechSupport"]=="No")).astype(int)

In [ ]:
telecom["has_family"]=((telecom["Partner"]=="Yes")|(telecom["Dependents"]=="Yes")).astype(int)

In [ ]:
telecom[service_cols]=telecom[service_cols].applymap(lambda x:1 if x=="Yes" else 0)

/tmp/ipython-input-1111230187.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  telecom[service_cols]=telecom[service_cols].applymap(lambda x:1 if x=="Yes" else 0)


In [ ]:
telecom["charges_ratio"]=telecom["MonthlyCharges"]/(telecom["TotalCharges"]+1)
telecom["tenure_charge_ratio"]=telecom["tenure"]/telecom["MonthlyCharges"]

In [ ]:
telecom["Churn"]=telecom["Churn"].map({"Yes":1,"No":0})

In [ ]:
telecom["Partner"]=telecom["Partner"].map({"Yes":1,"No":0})

In [ ]:
telecom["Dependents"]=telecom["Dependents"].map({"Yes":1,"No":0})

In [ ]:
telecom["gender"]=telecom["gender"].map({"Male":1,"Female":0})

In [ ]:
telecom.drop("customerID",axis=1,inplace=True)

In [ ]:
telecom["PaperlessBilling"]=telecom["PaperlessBilling"].map({"Yes":1,"No":0})

In [ ]:
telecom["tenure_group"]=telecom["tenure_group"].cat.codes

In [ ]:
telecom["InternetService"]=telecom["InternetService"].map({"None":0,"DSL":1,"Fiber optic":2})

In [ ]:
telecom["Contract"]=telecom["Contract"].map({"Month-to-month":0,"One year":1,"Two year":2})

In [ ]:
telecom["PaymentMethod"]=telecom["PaymentMethod"].map({"Electronic check":0,"Mailed check":1,"Bank transfer (automatic)":2,"Credit card (automatic)":3})

In [ ]:
telecom.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,tenure_group,monthly_avg,is_autopay,num_services,short_contract,is_fiber,tech_risk,has_family,charges_ratio,tenure_charge_ratio
0,0,0,1,0,1,0,0,1.0,0,1,...,0,29.850000,0,1,0,0,1,1,0.967585,0.033501
1,1,0,0,0,34,1,0,1.0,1,0,...,2,55.573529,0,3,0,0,1,0,0.030124,0.597015
2,1,0,0,0,2,1,0,1.0,1,1,...,0,54.075000,0,3,0,0,1,0,0.493358,0.037140
3,1,0,0,0,45,0,0,1.0,1,0,...,2,40.905556,0,3,0,0,0,0,0.022967,1.063830
4,0,0,0,0,2,1,0,2.0,0,0,...,0,75.825000,0,1,0,1,1,0,0.463151,0.028289


In [ ]:
telecom["TechSupport"].value_counts()

,count
TechSupport,
0,4999
1,2044


In [ ]:
telecom["tenure_group"].values

array([0, 2, 0, ..., 1, 0, 2], dtype=int8)

In [ ]:
Y=telecom["Churn"]
X=telecom.drop("Churn",axis=1)
X_train,X_temp,Y_train,Y_temp=train_test_split(X,Y,test_size=0.2,random_state=42,stratify=Y)
X_val,X_test,Y_val,Y_test=train_test_split(X_temp,Y_temp,test_size=0.2,stratify=Y_temp,random_state=42)

Model

In [ ]:
!pip install xgboost
from xgboost import XGBClassifier

In [ ]:
xgbmodel=XGBClassifier(n_estimators=300,max_depth=5,learning_rate=0.2,subsample=0.8,colsample_bytree=0.8,eval_metric="auc",objective="binary:logistic",
                       early_stopping_rounds=30,random_state=42)
xgbmodel.fit(X_train,Y_train,eval_set=[(X_val,Y_val)],verbose=False)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=30,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.2, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
#prediction
ypred=xgbmodel.predict(X_test)
yprob=xgbmodel.predict_proba(X_test)[:,1]

Insights

In [ ]:
from sklearn.metrics import confusion_matrix,classification_report,roc_auc_score
print("confusion Marix\n",confusion_matrix(Y_test,ypred))
print("Classification Report\n",classification_report(Y_test,ypred))
print("Ability to clssify\n",roc_auc_score(Y_test,yprob))

confusion Marix
 [[187  20]
 [ 39  36]]
Classification Report
               precision    recall  f1-score   support

           0       0.83      0.90      0.86       207
           1       0.64      0.48      0.55        75

    accuracy                           0.79       282
   macro avg       0.74      0.69      0.71       282
weighted avg       0.78      0.79      0.78       282

Ability to clssify
 0.8341384863123993


In [ ]:
#important features
imp_features=pd.Series(xgbmodel.feature_importances_,index=X.columns).sort_values(ascending=False)
print(imp_features)

Contract               0.283675
InternetService        0.163837
is_fiber               0.112681
tech_risk              0.069804
StreamingMovies        0.042184
tenure_charge_ratio    0.027927
PhoneService           0.026463
tenure                 0.022994
OnlineSecurity         0.022450
PaymentMethod          0.019959
PaperlessBilling       0.019845
MultipleLines          0.017942
monthly_avg            0.016703
MonthlyCharges         0.015844
TotalCharges           0.015118
charges_ratio          0.015073
OnlineBackup           0.012628
Dependents             0.012577
StreamingTV            0.010083
SeniorCitizen          0.009971
DeviceProtection       0.009970
gender                 0.009357
Partner                0.009205
has_family             0.009154
tenure_group           0.008926
num_services           0.007859
TechSupport            0.007768
short_contract         0.000000
is_autopay             0.000000
dtype: float32


In [ ]:
final_df=X_test.copy()
final_df["Actual Churn"]=Y_test
final_df["Predicted Churn"]=ypred
final_df["Predicted Probability of Churn"]=yprob
final_df.to_csv("Churn_Prediction_dataset.csv",index=False)
from google.colab import files
files.download("Churn_Prediction_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
final_df.to_csv("Churn_Prediction.pkl",index=False)

In [ ]:
import pickle as pk
pk.dump(xgbmodel,open("Churn_Predict.pkl","wb"))
files.download("Churn_Predict.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>